<a href="https://colab.research.google.com/github/lmendres/Test/blob/main/cases/case1_rct_retention/case1_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case 1 · Do win-back discounts work? (Part 2)
**BUS-G 492 · Causal decisions with AI**

In [1]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case1_rct_retention/data/"
for candidate in ["data/", "cases/case1_rct_retention/data/", "../../../cases/case1_rct_retention/data/", "../../../../cases/case1_rct_retention/data/"]:
    if os.path.exists(candidate + "customers_history.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

Reading data from: https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case1_rct_retention/data/


## Stage 5 · Something you didn't know

> **From:** Luis Ortega, Retention Team Lead
> **Re:** how agents decide who gets the offer
>
> We don't pick at random. In the CRM we can see who visited the **cancel page** in the last week and what they said in **chat**. If someone is clearly on their way out, we send the offer. None of that makes it into the analytics warehouse; the risk score doesn't see it either.
>
> Also, you may not know this: in **April 2026 we ran a test**. Among 8,000 customers with risk_score ≥ 30, a **coin flip** decided who got the offer. The results are in `april_discount_test.csv`.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. This cell loads the libraries, the Part 1 data (needed in Step 4) and the April test.

In [2]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
df = pd.read_csv(DATA + "customers_history.csv")
test = pd.read_csv(DATA + "april_discount_test.csv")
print(df.shape, test.shape)
test.head()

(24000, 14) (8000, 13)


,customer_id,region,plan,tenure_months,orders_last_90d,avg_order_value,skipped_weeks_90d,support_tickets_90d,app_sessions_per_week,risk_score,assigned_discount,churned_90d,revenue_next_90d
0,100000,Midwest,2-person,29,8,68.41,3,0,2.7,37,0,0,506.58
1,100001,Northeast,family,27,3,85.68,0,0,0.7,39,1,1,16.61
2,100002,Midwest,family,2,6,90.24,0,3,3.9,63,1,0,402.62
3,100003,South,family,13,9,103.06,2,1,2.4,55,1,1,80.70
4,100004,West,2-person,16,7,59.94,2,0,0.3,52,0,1,7.40


## Your tasks

Use AI again, but this time **tell it the whole story**: how the data were generated, and what the new information is. Each step ends with a table: fill it in (double-click the cell to type) before moving on.

### Step 1 · Check the coin flip

Compare the two groups on everything measured *before* the offer (tenure, orders, risk score, etc.). Should they differ?

| Measured before the offer | No offer | Offer | T-stat |
| --- | --- | --- | --- |
| tenure_months |12.997 |12.992 |0.028 |
| orders_last_90d | | | |
| skipped_weeks_90d | | | |
| support_tickets_90d | | | |
| app_sessions_per_week | | | |
| risk_score | | | |

In [4]:
# Your code here
from scipy.stats import ttest_ind

variables = [
    "tenure_months",
    "orders_last_90d",
    "skipped_weeks_90d",
    "support_tickets_90d",
    "app_sessions_per_week",
    "risk_score"
]

# Check the column names to identify the offer assignment variable
print(df.columns.tolist())


['customer_id', 'quarter', 'region', 'plan', 'tenure_months', 'orders_last_90d', 'avg_order_value', 'skipped_weeks_90d', 'support_tickets_90d', 'app_sessions_per_week', 'risk_score', 'got_discount', 'churned_90d', 'revenue_next_90d']


In [7]:
from scipy.stats import ttest_ind

variables = [
    "tenure_months",
    "orders_last_90d",
    "skipped_weeks_90d",
    "support_tickets_90d",
    "app_sessions_per_week",
    "risk_score"
]

no_offer = test[test["assigned_discount"] == 0]
offer = test[test["assigned_discount"] == 1]

print("No-offer customers:", len(no_offer))
print("Offer customers:", len(offer))

results = []

for col in variables:
    t_stat, p_value = ttest_ind(
        no_offer[col].dropna(),
        offer[col].dropna(),
        equal_var=False
    )

    results.append({
        "Measured before the offer": col,
        "No offer": round(no_offer[col].mean(), 3),
        "Offer": round(offer[col].mean(), 3),
        "T-stat": round(t_stat, 3)
    })

display(pd.DataFrame(results))


No-offer customers: 4064
Offer customers: 3936


,Measured before the offer,No offer,Offer,T-stat
0,tenure_months,12.997,12.992,0.028
1,orders_last_90d,6.467,6.442,0.473
2,skipped_weeks_90d,2.487,2.471,0.479
3,support_tickets_90d,0.688,0.685,0.167
4,app_sessions_per_week,2.047,2.053,-0.192
5,risk_score,64.448,64.476,-0.061


In [6]:
print(df["got_discount"].value_counts())
print(test.columns.tolist())
print(test.shape)

got_discount
0    19809
1     4191
Name: count, dtype: int64
['customer_id', 'region', 'plan', 'tenure_months', 'orders_last_90d', 'avg_order_value', 'skipped_weeks_90d', 'support_tickets_90d', 'app_sessions_per_week', 'risk_score', 'assigned_discount', 'churned_90d', 'revenue_next_90d']
(8000, 13)


### Step 2 · Effect on the treated (ETT)

Estimate the effect of the offer on churn **for the customers who received it**, with a 95% confidence interval. In this test every customer assigned the offer received it, so the ETT equals the simple difference in churn by assignment (the intention-to-treat effect, ITT).

| | Value |
|---|---|
| Churn rate, offer | |
| Churn rate, no offer | |
| ETT (percentage points) | |
| 95% confidence interval | |

Why is a simple difference in means enough here? How would you get the ETT if only some of the assigned customers had actually received the offer?

In [8]:
# Your code here
from statsmodels.stats.proportion import confint_proportions_2indep

# Separate customers by discount assignment
offer = test[test["assigned_discount"] == 1]
no_offer = test[test["assigned_discount"] == 0]

# Count customers who churned
churn_offer = offer["churned_90d"].sum()
churn_no_offer = no_offer["churned_90d"].sum()

# Group sizes
n_offer = len(offer)
n_no_offer = len(no_offer)

# Churn rates
rate_offer = churn_offer / n_offer
rate_no_offer = churn_no_offer / n_no_offer

# Effect: offer minus no offer
ett = rate_offer - rate_no_offer

# 95% confidence interval for the difference in proportions
ci_low, ci_high = confint_proportions_2indep(
    churn_offer, n_offer,
    churn_no_offer, n_no_offer,
    method="newcomb",
    compare="diff",
    alpha=0.05
)

print(f"Churn rate, offer:       {rate_offer:.4%}")
print(f"Churn rate, no offer:    {rate_no_offer:.4%}")
print(f"ETT (percentage points): {ett * 100:.3f}")
print(f"95% confidence interval: [{ci_low * 100:.3f}, {ci_high * 100:.3f}] percentage points")


Churn rate, offer:       29.9543%
Churn rate, no offer:    41.2156%
ETT (percentage points): -11.261
95% confidence interval: [-13.336, -9.172] percentage points


### Step 3 · Who benefits? ETT by risk band

Split the test customers into risk-score bands and estimate the ETT within each band (the coin flip was fair inside every band too).

Recall profits (value per customer) = $-(Y_{1i}-Y_{0i})*\$800 - ((1-Y_{1i})*\$90 + Y_{1i}*\$20$

| Risk band | Churn, no offer | Churn, offer | ETT = drop in churn (points) | Value per customer |
|---|---|---|---|---|
| 30–49 | | | | |
| 50–69 | | | | |
| 70–84 | | | | |
| 85–100 | | | | |

In [9]:
test["band"] = pd.cut(test.risk_score, [29, 49, 69, 84, 100], labels=["30-49", "50-69", "70-84", "85-100"])
# Your code here
# Define the risk bands
bins = [30, 50, 70, 85, 101]
labels = ["30–49", "50–69", "70–84", "85–100"]

test["risk_band"] = pd.cut(
    test["risk_score"],
    bins=bins,
    labels=labels,
    right=False
)

results = []

for band in labels:
    group = test[test["risk_band"] == band]

    no_offer = group[group["assigned_discount"] == 0]
    offer = group[group["assigned_discount"] == 1]

    # Churn rates
    churn_no = no_offer["churned_90d"].mean()
    churn_offer = offer["churned_90d"].mean()

    # Drop in churn, in percentage points
    ett_drop = (churn_no - churn_offer) * 100

    # Expected value per customer using the assignment's formula
    value = (
        (churn_no - churn_offer) * 800
        - ((1 - churn_offer) * 90 + churn_offer * 20)
    )

    results.append({
        "Risk band": band,
        "Churn, no offer (%)": round(churn_no * 100, 2),
        "Churn, offer (%)": round(churn_offer * 100, 2),
        "ETT = drop in churn (points)": round(ett_drop, 2),
        "Value per customer ($)": round(value, 2)
    })

display(pd.DataFrame(results))


,Risk band,"Churn, no offer (%)","Churn, offer (%)",ETT = drop in churn (points),Value per customer ($)
0,30–49,33.22,23.73,9.49,2.54
1,50–69,38.80,28.08,10.72,15.43
2,70–84,43.19,30.82,12.38,30.59
3,85–100,53.18,39.88,13.30,44.32


### Step 4 · Choose T


**Choose T** (offer to every band where the value is positive). **Predicted profit:** next quarter's 30,000 customers have about the same risk mix as `customers_history.csv`, so customers in a band ≈ 30,000 × that band's share in `df`. Predicted profit = Σ (value per customer × customers) over the bands you offer.


The test only included customers with risk_score ≥ 30. What can you say, and not say, about customers below 30? Answer in two or three sentences.

In [11]:
# The rule applies to ALL next-quarter customers, not just the test group (which only has risk >= 30).
# Their risk mix comes from customers_history.csv:
df = pd.read_csv(DATA + "customers_history.csv")
df["band"] = pd.cut(df.risk_score, [0, 29, 49, 69, 84, 100], labels=["below 30", "30-49", "50-69", "70-84", "85-100"])
print(df.band.value_counts(normalize=True).sort_index())   # share of customers in each band
# Your code here
# Risk-band shares among historical customers
# Risk-band shares in the historical customer population
df = pd.read_csv(DATA + "customers_history.csv")

df["band"] = pd.cut(
    df["risk_score"],
    bins=[0, 30, 50, 70, 85, 101],
    labels=["below 30"] + labels,
    right=False
)

shares = df["band"].value_counts(normalize=True)

total_profit = 0
rows = []

for band in labels:
    group = test[test["band"] == band]

    no_offer = group[group["assigned_discount"] == 0]
    offer = group[group["assigned_discount"] == 1]

    y0 = no_offer["churned_90d"].mean()
    y1 = offer["churned_90d"].mean()

    # Expected value per customer, following the assignment formula
    value = -(y1 - y0) * 800 - ((1 - y1) * 90 + y1 * 20)

    # Projected customer count next quarter
    customers = 30000 * shares.get(band, 0)

    # Offer only if value is positive
    offer_discount = value > 0
    profit = value * customers if offer_discount else 0
    total_profit += profit

    rows.append({
        "Risk band": band,
        "Historical share (%)": round(shares.get(band, 0) * 100, 2),
        "Value per customer ($)": round(value, 2),
        "Projected customers": round(customers),
        "Offer discount?": offer_discount,
        "Predicted profit ($)": round(profit, 2)
    })

display(pd.DataFrame(rows))
print(f"TOTAL PREDICTED PROFIT: ${total_profit:,.2f}")


band
below 30    0.294958
30-49       0.200000
50-69       0.200000
70-84       0.150042
85-100      0.155000
Name: proportion, dtype: float64


,Risk band,Historical share (%),Value per customer ($),Projected customers,Offer discount?,Predicted profit ($)
0,30–49,20.0,NaN,6000,False,0
1,50–69,20.0,NaN,6000,False,0
2,70–84,15.0,NaN,4501,False,0
3,85–100,15.5,NaN,4650,False,0


TOTAL PREDICTED PROFIT: $0.00


### Decide again

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
RISK_THRESHOLD = None      # offer goes to customers with risk_score >= this; 101 = nobody  (e.g. 60)
PREDICTED_PROFIT = None    # in dollars, as a number, e.g. 150000
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""

# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("RISK_THRESHOLD", RISK_THRESHOLD), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              1")
print("round:             2  (Part II)")
print(f"decision:          {RISK_THRESHOLD}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

## Submit (round 2)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Predicted profit:** a number such as `150000` (`$150,000` and `150K` also work, but not words).
- **Case:** `1`.  **Round:** `2` (Part II).
- **Submissions are final.** You can't change an answer after you submit, so check the printed values first. If your team submits more than once for the same round, only the **first** submission counts.
- Use the **same team name** in every round.

## Reflection (submit)

1. Your Part 1 analysis may have fit the data very well. Why was it still the wrong tool for this decision?
2. In one sentence: what was the difference between **predicting** the outcome and **deciding** what to do?
3. Name one other business decision where the same trap could appear.